In [1]:
import math

In [2]:
import os

In [6]:
from langgraph.graph import StateGraph, END

In [9]:
# 1. Define the state (just a list of messages)
workflow = StateGraph(dict)


In [10]:
def function_1(input_1):
    return input_1 + " First Function "

def function_2(input_2):
    return input_2 + "to Second Function"

In [11]:
workflow.add_node("node_1", function_1)
workflow.add_node("node_2", function_2)

workflow.add_edge('node_1', 'node_2')

workflow.set_entry_point("node_1")
workflow.set_finish_point("node_2")

In [12]:
app = workflow.compile()

In [13]:
app.invoke("i am moviing from")

'i am moviing from First Function to Second Function'

In [14]:
input = 'I am moving from'
for output in app.stream(input):
    # stream() yields dictionaries with output keyed by node name
    for key, value in output.items():
        print(f"Output from node '{key}':")
        print("---")
        print(value)
    print("\n---\n")

Output from node 'node_1':
---
I am moving from First Function 

---

Output from node 'node_2':
---
I am moving from First Function to Second Function

---



In [16]:
from langchain_groq import ChatGroq
llm=ChatGroq(model_name="llama3-groq-70b-8192-tool-use-preview")
llm.invoke("hi")

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [18]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# Define the structure of our state
class GraphState(TypedDict):
  """
  Represents the state of our graph.
  Attributes:
    message: The welcome message.
    question: The follow-up question.
  """
  message: str
  question: str

# Define the node functions
def welcome(state: GraphState) -> dict:
    """Adds a welcome message to the state."""
    state["message"] = "Welcome to LangGraph!"
    return state

def question(state: GraphState) -> dict:
    """Adds a follow-up message to the state."""
    state["question"] = "How can I help you today?"
    return state

def display(state: GraphState) -> dict:
    """Retrieves messages and prints them to the console."""
    print(state["message"])
    print(state["question"])
    return {}  # This node doesn't modify the state


In [19]:
# The graph needs to know what the state schema is
graph = StateGraph(GraphState)

# Add the three functions as named nodes
graph.add_node("welcome", welcome)
graph.add_node("question", question)
graph.add_node("display", display)


In [21]:
graph.set_entry_point("welcome")
graph.add_edge("welcome", "question")
graph.add_edge("question", "display")
graph.add_edge("display", END)


In [22]:
# This creates the runnable app
app = graph.compile()

# Run the app with an empty initial state
app.invoke({})


Welcome to LangGraph!
How can I help you today?


{'message': 'Welcome to LangGraph!', 'question': 'How can I help you today?'}

In [24]:
# Define the state for this conditional graph
class ConditionalState(TypedDict):
  input: str # User's query
  next: str  # The next node to route to

# Define the functions
def check_query(state: ConditionalState) -> dict:
    """Checks the query and returns a routing decision."""
    query = state.get("input", "").lower()
    if "pricing" in query:
      return {"next": "pricing"}
    else:
      return {"next": "general"}

def pricing_info(state: ConditionalState) -> dict:
    """Provides pricing information."""
    print("Here’s detailed pricing information.")
    return {}

def general_info(state: ConditionalState) -> dict:
    """Provides general information."""
    print("Here’s some general information.")
    return {}


In [25]:
# Set up the graph
graph = StateGraph(ConditionalState)
graph.add_node("check", check_query)
graph.add_node("pricing", pricing_info)
graph.add_node("general", general_info)

In [26]:
# Set the entry point and the conditional edges
graph.set_entry_point("check")
graph.add_conditional_edges(
  "check", # The source node for the decision
  lambda state: state["next"], # A function to extract the routing key from the state
  {
    # A map of routing keys to destination nodes
    "pricing": "pricing",
    "general": "general"
  }
)


In [27]:
# Add edges from the final nodes to the end
graph.add_edge("pricing", END)
graph.add_edge("general", END)

# Compile and run
app = graph.compile()
app.invoke({"input": "I need to know about your pricing plans."})


Here’s detailed pricing information.


{'input': 'I need to know about your pricing plans.', 'next': 'pricing'}